In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Team Names"
FILES = {
    "serie_a":        f"{BASE}/serie_a_teams.csv",
    "premier_league": f"{BASE}/premier_league_teams.csv",
    "la_liga":        f"{BASE}/la_liga_teams.csv",
    "bundesliga":     f"{BASE}/bundesliga_teams.csv",
}

In [41]:
def load_league(path, league):
  df = pd.read_csv(path, dtype=str, low_memory=False)
  cols = {c.lower(): c for c in df.columns}

  std_lc = next((k for k in ["team_standardized"] if k in cols), None)
  full_lc = next((k for k in ["team_full_name"] if k in cols), None)
  season_lc = next((k for k in ["season"] if k in cols), None)

  if std_lc is None or full_lc is None:
        raise ValueError(f"{league}: expected team standardized/full columns not found in {path} "
                         f"(got columns: {list(df.columns)})")

  rename_map = {cols[std_lc]: "team_std", cols[full_lc]: "team_full", }

  if season_lc is not None:
    rename_map[cols[season_lc]] = "season"

  df = df.rename(columns=rename_map)
  if "season" not in df.columns:
    df["season"] = pd.NA

  df["team_std"]  = df["team_std"].str.strip()
  df["team_full"] = df["team_full"].str.strip()

  df = df[["season","team_std","team_full"]].drop_duplicates().copy()
  df.insert(0, "league", league)
  return df

In [42]:
per_league = {lg: load_league(path, lg) for lg, path in FILES.items()}
combined = pd.concat(per_league.values(), ignore_index=True)
display(combined.sample(n=5))

,league,season,team_std,team_full
1392,la_liga,2015,GET,Getafe
62,serie_a,2001,VEN,Venezia
1347,la_liga,2013,ATHM,Ath Madrid
573,serie_a,2023,CAG,Cagliari
1698,bundesliga,2006,BIE,Bielefeld


In [43]:
combined["_season_num"] = pd.to_numeric(combined["season"], errors="coerce")

sorted_master = (
    combined
    .sort_values(
        ["league", "_season_num", "team_std"],
        ascending=[True, False, True],
        kind="mergesort"
    )
    .drop(columns=["_season_num"])
    .reset_index(drop=True)
)

display(sorted_master.head(20))

,league,season,team_std,team_full
0,bundesliga,2024,AUG,Augsburg
1,bundesliga,2024,BAY,Bayern Munich
2,bundesliga,2024,BOC,Bochum
3,bundesliga,2024,DOR,Dortmund
4,bundesliga,2024,EINF,Ein Frankfurt
5,bundesliga,2024,FRE,Freiburg
6,bundesliga,2024,HEI,Heidenheim
7,bundesliga,2024,HOF,Hoffenheim
8,bundesliga,2024,HOLK,Holstein Kiel
9,bundesliga,2024,LEI,RB Leipzig


In [45]:
# (b) Checking if same team_full (full team name) repeated within the same league & season
dups_full_per_season = (
    combined[combined.duplicated(["league","season","team_full"], keep=False)]
    .sort_values(["league","season","team_full","team_std"])
)
print(f"[Within-league/season] duplicate team_full rows: {len(dups_full_per_season)}")

display(dups_full_per_season.head(50))

[Within-league/season] duplicate team_full rows: 0


,league,season,team_std,team_full,_season_num


In [48]:
# Checking duplicate standardized names (within a league)

dups_std_per_season = (
    combined[combined.duplicated(["league","season","team_std"], keep=False)]
    .sort_values(["league","season","team_std","team_full"])
)
print(f"[Within-league/season] duplicate team_std rows: {len(dups_std_per_season)}")
display(dups_std_per_season.head(20))


[Within-league/season] duplicate team_std rows: 74


,league,season,team_std,team_full,_season_num
1112,la_liga,2000,MAL,Malaga,2000
1113,la_liga,2000,MAL,Mallorca,2000
1120,la_liga,2000,VAL,Valencia,2000
1121,la_liga,2000,VAL,Valladolid,2000
1122,la_liga,2000,VAL,Vallecano,2000
1133,la_liga,2001,MAL,Malaga,2001
1134,la_liga,2001,MAL,Mallorca,2001
1140,la_liga,2001,VAL,Valencia,2001
1141,la_liga,2001,VAL,Valladolid,2001
1142,la_liga,2001,VAL,Vallecano,2001


In [51]:
# New map for the same standard name within a league (similiar std name 2 teams within the same league)

combined["league"] = combined["league"].astype(str).str.strip()
combined["team_full"] = combined["team_full"].astype(str).str.strip()
combined["team_std"]  = combined["team_std"].astype(str).str.strip()
combined = combined.drop(columns=["_season_num"], errors="ignore")

fix_map = {
    ("la_liga", "Valencia"):     "VLC",
    ("la_liga", "Valladolid"):   "VLLD",
    ("la_liga", "Vallecano"):    "RAYO",
    ("la_liga", "Malaga"):       "MLG",
    ("la_liga", "Mallorca"):     "MALL",
    ("premier_league", "Blackburn"): "BLBN",
    ("premier_league", "Blackpool"): "BLPL",
}

for (lg, full_name), new_std in fix_map.items():
    mask = (combined["league"] == lg) & (combined["team_full"] == full_name)
    combined.loc[mask, "team_std"] = new_std

In [53]:
dups_after = combined[combined.duplicated(["league","season","team_std"], keep=False)] \
                    .sort_values(["league","season","team_std","team_full"])
print(f"[After fixes] duplicate team_std rows: {len(dups_after)}")
display(dups_after.head(20))

[After fixes] duplicate team_std rows: 0


,league,season,team_std,team_full


In [56]:
# Checking for multiple team_stds within the same league

full_to_std_incons = (
    combined.groupby(["league","team_full"])["team_std"].nunique().reset_index(name="n_std")
)
full_to_std_incons = full_to_std_incons[full_to_std_incons["n_std"] > 1]
print(f"[Within-league] team_full mapping to >1 standardized name: {len(full_to_std_incons)} rows")
display(
    combined.merge(full_to_std_incons[["league","team_full"]], on=["league","team_full"], how="inner")
            .sort_values(["league","team_full","team_std","season"])
)

[Within-league] team_full mapping to >1 standardized name: 0 rows


,league,season,team_std,team_full


In [59]:
# Checking same name across leagues

cross = (combined.drop_duplicates(["league","team_std"])
         .groupby("team_std")["league"].nunique().sort_values(ascending=False))
colliding_std = cross[cross > 1].index.tolist()

print(f"[Across leagues] team_std names used in >1 league: {len(colliding_std)}")
usage = (combined[combined["team_std"].isin(colliding_std)]
         .groupby(["team_std","league"])["team_full"]
         .unique()
         .reset_index()
         .sort_values(["team_std","league"]))
display(usage)

[Across leagues] team_std names used in >1 league: 9


,team_std,league,team_full
0,BAR,la_liga,[Barcelona]
1,BAR,serie_a,[Bari]
2,BOL,premier_league,[Bolton]
3,BOL,serie_a,[Bologna]
4,BRE,premier_league,[Brentford]
5,BRE,serie_a,[Brescia]
6,CAR,premier_league,[Cardiff]
7,CAR,serie_a,[Carpi]
8,HER,bundesliga,[Hertha]
9,HER,la_liga,[Hercules]


In [61]:
# Cleaning cross-league similar names

combined = combined.drop(columns=["_season_num"], errors="ignore").copy()

fix_map = {
    ("la_liga",        "Barcelona"):  "BARC",
    ("serie_a",        "Bari"):       "BARI",

    ("premier_league", "Bolton"):     "BOLN",
    ("serie_a",        "Bologna"):    "BOLG",

    ("premier_league", "Brentford"):  "BRNT",
    ("serie_a",        "Brescia"):    "BRES",

    ("premier_league", "Cardiff"):    "CARD",
    ("serie_a",        "Carpi"):      "CARP",

    ("bundesliga",     "Hertha"):     "HERT",
    ("la_liga",        "Hercules"):   "HERC",

    ("bundesliga",     "RB Leipzig"): "RBL",
    ("premier_league", "Leicester"):  "LEIC",

    ("bundesliga",     "Leverkusen"): "LEVK",
    ("la_liga",        "Levante"):    "LEVA",

    ("premier_league", "Liverpool"):  "LIVP",
    ("serie_a",        "Livorno"):    "LIVO",

    ("bundesliga",     "Wolfsburg"):  "WOB",
    ("premier_league", "Wolves"):     "WOLV",
}

for (lg, full_name), new_std in fix_map.items():
    mask = (combined["league"] == lg) & (combined["team_full"] == full_name)
    combined.loc[mask, "team_std"] = new_std

In [62]:
# Checking for cross-league similar names after fixes

cross = (combined.drop_duplicates(["league","team_std"])
         .groupby("team_std")["league"].nunique().sort_values(ascending=False))
colliding_std = cross[cross > 1].index.tolist()
print(f"[Across leagues] team_std names used in >1 league (after fixes): {len(colliding_std)}")
if colliding_std:
    display(
        combined[combined["team_std"].isin(colliding_std)]
                .sort_values(["team_std","league","season"])
                [["team_std","league","season","team_full"]]
                .drop_duplicates()
                .head(100)
    )

[Across leagues] team_std names used in >1 league (after fixes): 0


In [63]:
# Printing out final team name table

combined["_season_num"] = pd.to_numeric(combined["season"], errors="coerce")
sorted_master = (
    combined.sort_values(["league","_season_num","team_std"],
                         ascending=[True, False, True], kind="mergesort")
            .drop(columns=["_season_num"])
            .reset_index(drop=True)
)

display(sorted_master.head(20))

,league,season,team_std,team_full
0,bundesliga,2024,AUG,Augsburg
1,bundesliga,2024,BAY,Bayern Munich
2,bundesliga,2024,BOC,Bochum
3,bundesliga,2024,DOR,Dortmund
4,bundesliga,2024,EINF,Ein Frankfurt
5,bundesliga,2024,FRE,Freiburg
6,bundesliga,2024,HEI,Heidenheim
7,bundesliga,2024,HOF,Hoffenheim
8,bundesliga,2024,HOLK,Holstein Kiel
9,bundesliga,2024,LEVK,Leverkusen


In [64]:
out_path = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Team Names/master_team_names.csv"
sorted_master.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Team Names/master_team_names.csv
